In [1]:
import pandas as pd   
import matplotlib.pyplot as plt
import numpy as np
import gc 
import seaborn as sns
import warnings
import lightgbm as lgb
from sklearn.metrics import mean_squared_log_error

#### Create frames and merge them

In [2]:
train = pd.read_csv("../data/train.csv", parse_dates=["timestamp"])
test = pd.read_csv("../data/test.csv", parse_dates=["timestamp"])
weather_train = pd.read_csv("../data/weather_train.csv", parse_dates=["timestamp"])
weather_test = pd.read_csv("../data/weather_test.csv", parse_dates=["timestamp"])
building_metadata = pd.read_csv("../data/building_metadata.csv")

train = train.merge(building_metadata, on='building_id', how='left')
test = test.merge(building_metadata, on='building_id', how='left')
train = train.merge(weather_train, on=['site_id', 'timestamp'], how='left')
test = test.merge(weather_test, on=['site_id', 'timestamp'], how='left')

train["timestamp"] = pd.to_datetime(train["timestamp"])
test["timestamp"] = pd.to_datetime(test["timestamp"])


### Memory reduction        

In [3]:
import pandas as pd
from pandas.api.types import (
    is_datetime64_any_dtype,
    is_categorical_dtype,
    is_integer_dtype,
    is_float_dtype,
    is_object_dtype,
    is_string_dtype,
)

def reduce_mem_usage(df, use_float16=False):
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory usage of dataframe is {start_mem:.2f} MB")

    for col in df.columns:
        s = df[col]
        dtype = s.dtype

        if is_datetime64_any_dtype(s) or is_categorical_dtype(dtype):
            continue

        if is_integer_dtype(dtype):
            df[col] = pd.to_numeric(s, downcast="integer")

        elif is_float_dtype(dtype):
            if use_float16:
                c_min = s.min()
                c_max = s.max()
                if c_min >= np.finfo(np.float16).min and c_max <= np.finfo(np.float16).max:
                    df[col] = s.astype(np.float16)
                else:
                    df[col] = pd.to_numeric(s, downcast="float")
            else:
                df[col] = pd.to_numeric(s, downcast="float")

        elif is_object_dtype(dtype) or is_string_dtype(dtype):
            df[col] = s.astype("category")

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory usage after optimization is: {end_mem:.2f} MB")
    print(f"Decreased by {100 * (start_mem - end_mem) / start_mem:.1f}%")

    return df

In [ ]:
reduce_mem_usage(train)
reduce_mem_usage(test)

In [ ]:
del weather_train, weather_test, building_metadata
gc.collect()

In [ ]:
def check_columns_are_same(df1, df2):
    
    cols1 = set(df1.columns)
    cols2 = set(df2.columns)
    if cols1 == cols2:
        print("Columns are the same in both dataframes.")
    else:
        print("Columns differ between the two dataframes.")
        print(f"Columns in train but not in test:", cols1 - cols2)
        print(f"Columns in test but not in train:", cols2 - cols1)

In [ ]:
check_columns_are_same(train, test)

#### Feature Engineering 

In [ ]:
## Drop some features that are not useful for modeling
train["building_age"] = train["timestamp"].dt.year - train["year_built"]
test["building_age"] = test["timestamp"].dt.year - test["year_built"]

train.drop(columns=["floor_count","year_built","cloud_coverage","wind_direction"], inplace=True)
test.drop(columns=["floor_count","year_built","cloud_coverage","wind_direction"], inplace=True)


In [ ]:
## Fill missing values with mean 

train["building_age"] = train["building_age"].fillna(train["building_age"].mean())
test["building_age"] = test["building_age"].fillna(train["building_age"].mean())

train["wind_speed"] = train["wind_speed"].fillna(train["wind_speed"].mean())
test["wind_speed"] = test["wind_speed"].fillna(train["wind_speed"].mean())

train["precip_depth_1_hr"] = train["precip_depth_1_hr"].fillna(train["precip_depth_1_hr"].mean())
test["precip_depth_1_hr"] = test["precip_depth_1_hr"].fillna(train["precip_depth_1_hr"].mean())

train["sea_level_pressure"] = train["sea_level_pressure"].fillna(train["sea_level_pressure"].mean())
test["sea_level_pressure"] = test["sea_level_pressure"].fillna(train["sea_level_pressure"].mean())


In [ ]:
def interpolate_weather_by_site(df, columns):
    df.sort_values(["site_id", "timestamp"], inplace=True)

    for col in columns:
        df[col] = df.groupby("site_id")[col].transform(
            lambda s: s.interpolate(method="linear", limit_direction="both")
        )

    return df

interpolate_cols = [
    "air_temperature",
    "dew_temperature",
]

train = interpolate_weather_by_site(train, interpolate_cols)
test = interpolate_weather_by_site(test, interpolate_cols)

In [ ]:
train.isnull().sum() / len(train) * 100

In [ ]:
## Extract time-based features from timestamp

train["hour"] = train["timestamp"].dt.hour
train["day"] = train["timestamp"].dt.day
train["month"] = train["timestamp"].dt.month
train["year"] = train["timestamp"].dt.year

test["hour"] = test["timestamp"].dt.hour
test["day"] = test["timestamp"].dt.day
test["month"] = test["timestamp"].dt.month
test["year"] = test["timestamp"].dt.year



In [ ]:
## Drop timestamp as we have extracted features from it
train.drop(columns=["timestamp"], inplace=True)
test.drop(columns=["timestamp"], inplace=True)


### Model Training  

In [ ]:
train.columns

In [ ]:
feature_cols = [
    "building_id",
    "meter",
    "site_id",
    "primary_use",
    "square_feet",
    "air_temperature",
    "dew_temperature",
    "precip_depth_1_hr",
    "sea_level_pressure",
    "wind_speed",
    "building_age",
    "hour",
    "day",
    "month",
    "year",
]

target_col = "log_meter_reading"
cat_cols = ["building_id", "meter", "site_id", "primary_use"]

In [ ]:
train_sorted = train.sort_values(["year", "month", "day", "hour"]).reset_index(drop=True).copy()
train_sorted["log_meter_reading"] = np.log1p(train_sorted["meter_reading"])

for col in cat_cols:
    train_sorted[col] = train_sorted[col].astype("category")

folds = 4
min_train_fraction = 0.60
valid_fraction = (1 - min_train_fraction) / folds

cv_splits = []

for fold in range(folds):
    train_end = int(len(train_sorted) * (min_train_fraction + valid_fraction * fold))
    valid_end = int(len(train_sorted) * (min_train_fraction + valid_fraction * (fold + 1)))
    cv_splits.append((train_end, valid_end))


In [ ]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "seed": 42,
    "verbosity": -1,
}


### LightGBM model training

In [ ]:
models = []
fold_scores = []
fold_predictions = []

for fold, (train_end, valid_end) in enumerate(cv_splits, start=1):
    train_part = train_sorted.iloc[:train_end]
    valid_part = train_sorted.iloc[train_end:valid_end]

    X_train = train_part[feature_cols]
    y_train = train_part[target_col]

    X_valid = valid_part[feature_cols]
    y_valid = valid_part[target_col]

    train_dataset = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols, free_raw_data=False)
    valid_dataset = lgb.Dataset(X_valid, label=y_valid, categorical_feature=cat_cols, free_raw_data=False)

    print(f"Fold {fold}: train rows={len(train_part):,}, valid rows={len(valid_part):,}")

    model = lgb.train(
        params,
        train_dataset,
        valid_sets=[train_dataset, valid_dataset],
        valid_names=["train", "valid"],
        num_boost_round=1000,
        callbacks=[
            lgb.early_stopping(stopping_rounds=100),
            lgb.log_evaluation(100),
        ],
    )

    valid_pred_log = model.predict(X_valid, num_iteration=model.best_iteration)
    valid_pred = np.expm1(valid_pred_log)
    valid_pred = np.clip(valid_pred, 0, None)

    y_valid_raw = valid_part["meter_reading"]
    fold_rmsle = np.sqrt(mean_squared_log_error(y_valid_raw, valid_pred))

    print(f"Fold {fold} RMSLE: {fold_rmsle:.4f}")

    models.append(model)
    fold_scores.append(fold_rmsle)
    fold_predictions.append((valid_part, valid_pred))
    

print(f"Mean CV RMSLE: {np.mean(fold_scores):.4f}")
print(f"Std CV RMSLE: {np.std(fold_scores):.4f}")

In [ ]:
valid_part, valid_pred = fold_predictions[-1]
rmsle = fold_scores[-1]

print(f"Last Fold RMSLE: {rmsle:.4f}")
print(f"Mean CV RMSLE: {np.mean(fold_scores):.4f}")


### DEBUG Phase

In [ ]:
pred_df = valid_part[["meter_reading","meter"]].copy()
pred_df["pred_meter"] = valid_pred

pred_df.info()

In [ ]:
pred_df["pred_meter"].unique().size   

In [ ]:
pred_df.groupby("meter")["pred_meter"].nunique()

In [ ]:
sample_df = pred_df.sample(25, random_state=42).reset_index(drop=True)
sample_df["sample_id"] = sample_df.index.astype(str)
sample_df = sample_df.sort_values("meter_reading").reset_index(drop=True)

plt.figure(figsize=(12, 10))

for i, row in sample_df.iterrows():
    plt.plot(
        [row["meter_reading"], row["pred_meter"]],
        [i, i],
        color="gray",
        alpha=0.7,
        linewidth=2,
    )

plt.scatter(
    sample_df["meter_reading"],
    range(len(sample_df)),
    color="royalblue",
    s=80,
    label="Actual",
)

plt.scatter(
    sample_df["pred_meter"],
    range(len(sample_df)),
    color="tomato",
    s=80,
    label="Predicted",
)

plt.yticks(range(len(sample_df)), sample_df["sample_id"])
plt.xlabel("Meter Reading Value")
plt.ylabel("Sample")
plt.title("Difference between Actual and Predicted Meter Readings")
plt.legend()
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.show()

#### ROADMAP FOR IMPROVEMENTS 

- apply cross-validation 
- train model for each meter type